# Preparación, limpieza y estructuración de datos
**Proyecto:** Barreras para la actividad física – PulseUp
**Evidencia:** GA1-220501094-01-AA1-EV07 · **Autor:** Santiago Varela Peña · ADSO, Ficha 3223873

**Pregunta analítica:** ¿Qué barreras impiden a los adultos realizar actividad física de forma regular, qué factores
sociodemográficos y de salud se asocian con la inactividad, y cuáles de esas barreras puede abordar una aplicación móvil
de hábitos saludables como PulseUp?

**Punto de partida:** el diagnóstico de `notebooks/data_quality.ipynb` y `reports/data_quality.md`. Este notebook aplica las
lecciones 2 (limpieza), 3 (filtrado y agrupación) y 4 (combinación y estructuración) a los tres datasets del proyecto.
Los archivos de `data/raw/` no se modifican: los resultados se guardan en `data/processed/`.

## Preparación del entorno y carga de los datos originales

In [1]:
import re
import unicodedata
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import pyreadstat

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 200)

# En Google Colab los datos se leen desde Google Drive; en local, desde data/raw/
try:
    from google.colab import drive
    drive.mount("/content/drive")
    DATA_RAW = Path("/content/drive/MyDrive/pulseup-datasets")
    DATA_PROCESSED = DATA_RAW / "processed"
except ModuleNotFoundError:
    DATA_RAW = Path("..") / "data" / "raw"
    DATA_PROCESSED = Path("..") / "data" / "processed"
DATA_PROCESSED.mkdir(parents=True, exist_ok=True)

print("Pandas:", pd.__version__)
print("Datos originales en:", DATA_RAW)
print("Datos procesados en:", DATA_PROCESSED)

Pandas: 3.1.1
Datos originales en: ..\data\raw
Datos procesados en: ..\data\processed


In [2]:
def buscar_archivo(*patrones):
    """Devuelve el primer archivo de la carpeta de datos que coincida con alguno de los patrones."""
    for patron in patrones:
        encontrados = sorted(DATA_RAW.glob(patron))
        if encontrados:
            return encontrados[0]
    raise FileNotFoundError(f"No se encontró {patrones} en {DATA_RAW.resolve()}. Revisa data/raw/README.md.")


# Dataset 1: BRFSS 2023 (solo las variables del proyecto, leído por bloques)
VARIABLES_BRFSS = ["_STATE", "SEXVAR", "_AGEG5YR", "_AGE80", "EDUCA", "EMPLOY1", "INCOME3",
                   "EXERANY2", "_TOTINDA", "GENHLTH", "_BMI5", "ADDEPEV3", "_LLCPWT"]
bloques = []
with pd.read_sas(buscar_archivo("LLCP2023*", "*.XPT*", "*.xpt"), format="xport",
                 chunksize=50_000, encoding="latin-1") as lector:
    for bloque in lector:
        bloques.append(bloque[[c for c in VARIABLES_BRFSS if c in bloque.columns]])
brfss_raw = pd.concat(bloques, ignore_index=True)

# Dataset 2: actividad física en Bogotá (latin-1, separado por ";")
bogota_raw = pd.read_csv(buscar_archivo("*actividad*fisica*.csv", "*.csv"), encoding="latin-1", sep=";")

# Dataset 3: Eurobarómetro 97.3 (códigos numéricos + metadatos con las etiquetas)
ruta_euro = buscar_archivo("ZA7888*.dta", "ZA7888*.sav")
lector_euro = pyreadstat.read_dta if ruta_euro.suffix == ".dta" else pyreadstat.read_sav
euro_raw, meta_euro = lector_euro(ruta_euro)
ETIQUETAS_VARIABLES = meta_euro.column_names_to_labels
ETIQUETAS_VALORES = meta_euro.variable_value_labels

for nombre, df in [("BRFSS", brfss_raw), ("Bogotá", bogota_raw), ("Eurobarómetro", euro_raw)]:
    print(f"{nombre:14} {df.shape[0]:>8,} filas x {df.shape[1]:>3} columnas")

BRFSS           433,323 filas x  13 columnas
Bogotá              168 filas x   4 columnas
Eurobarómetro    26,569 filas x 501 columnas


# Lección 2. Limpieza y estandarización

Cada paso de limpieza es una **función reutilizable** que recibe un DataFrame y devuelve uno nuevo, sin modificar el original.
Las funciones se encadenan con `pipe()` y la función `registrar` anota el estado de los datos después de cada paso.

In [3]:
REGISTRO = []


def registrar(df, tabla, paso):
    """Anota filas, columnas, nulos y memoria después de un paso del pipeline."""
    REGISTRO.append({
        "tabla": tabla, "paso": paso, "filas": len(df), "columnas": df.shape[1],
        "nulos": int(df.isna().sum().sum()),
        "memoria_mb": round(df.memory_usage(deep=True).sum() / 1_048_576, 1),
    })
    return df


def normalizar_nombre(texto):
    """'Percepción ' -> 'percepcion': sin tildes, en minúsculas y con guiones bajos."""
    texto = unicodedata.normalize("NFKD", str(texto)).encode("ascii", "ignore").decode()
    return re.sub(r"[^a-z0-9]+", "_", texto.lower()).strip("_")


def limpiar_texto(df, columnas):
    """Quita espacios sobrantes y unifica la codificación Unicode de columnas de texto."""
    df = df.copy()
    for col in columnas:
        df[col] = (df[col].astype("string").str.normalize("NFC")
                   .str.replace(r"\s+", " ", regex=True).str.strip())
    return df

## Dataset 1: BRFSS 2023

### Estado inicial
El diagnóstico encontró códigos de "no sabe / no responde" guardados como números, tipos `float64` en variables categóricas,
el IMC multiplicado por 100 y 252 filas idénticas. Antes de limpiar se revisan esos problemas.

In [4]:
CODIGOS_ESPECIALES = {
    "EXERANY2": [7, 9], "_TOTINDA": [9], "GENHLTH": [7, 9], "ADDEPEV3": [7, 9],
    "EDUCA": [9], "EMPLOY1": [9], "INCOME3": [77, 99], "_AGEG5YR": [14],
}


def contar_codigos_especiales(df, codigos):
    return pd.Series({col: int(df[col].isin(c).sum()) for col, c in codigos.items() if col in df.columns},
                     name="registros con código especial")


print("Tipos:", brfss_raw.dtypes.value_counts().to_dict())
print("Nulos por columna:", brfss_raw.isna().sum()[lambda s: s > 0].to_dict())
contar_codigos_especiales(brfss_raw, CODIGOS_ESPECIALES).to_frame()

Tipos: {dtype('float64'): 13}
Nulos por columna: {'EDUCA': 9, 'EMPLOY1': 2968, 'INCOME3': 8075, 'EXERANY2': 2, 'GENHLTH': 4, '_BMI5': 40535, 'ADDEPEV3': 3}


,registros con código especial
EXERANY2,1249
_TOTINDA,1251
GENHLTH,1258
ADDEPEV3,2584
EDUCA,2316
EMPLOY1,4713
INCOME3,78548
_AGEG5YR,7779


In [5]:
# Duplicados: ¿son errores o personas distintas con las mismas respuestas?
duplicados = brfss_raw[brfss_raw.duplicated(keep=False)]
print("Filas idénticas (contando todas las copias):", len(duplicados))
print("Filas que drop_duplicates eliminaría:", brfss_raw.duplicated().sum())
print("Estados distintos entre los duplicados:", duplicados["_STATE"].nunique())
duplicados.sort_values(list(brfss_raw.columns)).head(6)

Filas idénticas (contando todas las copias): 504


Filas que drop_duplicates eliminaría: 252
Estados distintos entre los duplicados: 42


,_STATE,SEXVAR,_AGEG5YR,_AGE80,EDUCA,EMPLOY1,INCOME3,EXERANY2,_TOTINDA,GENHLTH,_BMI5,ADDEPEV3,_LLCPWT
1041,1.0,1.0,1.0,19.0,3.0,4.0,3.0,1.0,1.0,2.0,2712.0,1.0,3435.030925
3941,1.0,1.0,1.0,19.0,3.0,4.0,3.0,1.0,1.0,2.0,2712.0,1.0,3435.030925
1143,1.0,1.0,1.0,20.0,4.0,1.0,77.0,1.0,1.0,2.0,1825.0,2.0,3048.186669
4023,1.0,1.0,1.0,20.0,4.0,1.0,77.0,1.0,1.0,2.0,1825.0,2.0,3048.186669
1094,1.0,1.0,1.0,20.0,4.0,1.0,99.0,1.0,1.0,1.0,3254.0,2.0,1563.796957
3981,1.0,1.0,1.0,20.0,4.0,1.0,99.0,1.0,1.0,1.0,3254.0,2.0,1563.796957


**Decisión sobre duplicados:** el subconjunto no tiene un identificador del encuestado y en cada estado se asigna el mismo peso
muestral a personas del mismo grupo; por eso, dos personas distintas pueden tener las 13 respuestas iguales.
Eliminarlas borraría encuestados reales, así que **se conservan** y se crea un identificador `id_registro`.

In [6]:
# Consistencia entre EXERANY2 (respuesta del encuestado) y _TOTINDA (indicador calculado por el CDC)
pd.crosstab(brfss_raw["EXERANY2"], brfss_raw["_TOTINDA"], margins=True, dropna=False)

_TOTINDA,1.0,2.0,9.0,All
EXERANY2,,,,
1.0,325227,0,0,325227
2.0,0,106845,0,106845
7.0,0,0,927,927
9.0,0,0,322,322
NaN,0,0,2,2
All,325227,106845,1251,433323


**Decisión sobre `_TOTINDA`:** si la tabla muestra que coincide con `EXERANY2` (1 con 1 y 2 con 2), es una variable redundante.
Se conserva `EXERANY2` como variable objetivo y se descarta `_TOTINDA`. También se descarta `_AGEG5YR`, porque `_AGE80`
tiene la edad exacta sin faltantes y permite construir los grupos de edad del proyecto.

### Funciones de limpieza

In [7]:
SEXO = {1: "Hombre", 2: "Mujer"}
SI_NO = {1: "Sí", 2: "No"}
EDUCACION = {1: "Sin escolaridad", 2: "Primaria", 3: "Secundaria incompleta",
             4: "Secundaria completa", 5: "Universidad incompleta", 6: "Universidad completa"}
SITUACION_LABORAL = {1: "Empleado", 2: "Independiente", 3: "Desempleado 1 año o más",
                     4: "Desempleado menos de 1 año", 5: "Hogar", 6: "Estudiante",
                     7: "Jubilado", 8: "Incapacitado para trabajar"}
INGRESOS = {1: "Menos de 25 mil USD", 2: "Menos de 25 mil USD", 3: "Menos de 25 mil USD", 4: "Menos de 25 mil USD",
            5: "25 a 50 mil USD", 6: "25 a 50 mil USD", 7: "50 a 100 mil USD", 8: "50 a 100 mil USD",
            9: "100 a 200 mil USD", 10: "100 a 200 mil USD", 11: "200 mil USD o más"}
ORDEN_INGRESOS = ["Menos de 25 mil USD", "25 a 50 mil USD", "50 a 100 mil USD",
                  "100 a 200 mil USD", "200 mil USD o más", "No informa"]
SALUD = {1: "Excelente", 2: "Muy buena", 3: "Buena", 4: "Regular", 5: "Mala"}


def categoria(serie, mapa, orden=None, ordenada=True):
    """Convierte códigos numéricos en una categoría con etiquetas en español."""
    orden = orden or list(dict.fromkeys(mapa.values()))
    return pd.Series(pd.Categorical(serie.map(mapa), categories=orden, ordered=ordenada), index=serie.index)


def crear_identificador(df):
    df = df.copy()
    df.insert(0, "id_registro", np.arange(1, len(df) + 1, dtype="int32"))
    return df


def reemplazar_codigos_especiales(df, codigos):
    """Convierte los códigos de 'no sabe / no responde' en valores faltantes."""
    df = df.copy()
    for col, valores in codigos.items():
        if col in df.columns:
            df[col] = df[col].mask(df[col].isin(valores))
    return df


def excluir_sin_actividad(df):
    """Quita los registros sin respuesta válida sobre actividad física: sin ella no se sabe si la persona es activa."""
    return df[df["EXERANY2"].notna()].reset_index(drop=True)


def corregir_tipos_y_etiquetas(df):
    """Escala el IMC, corrige tipos y traduce los códigos a categorías en español."""
    return pd.DataFrame({
        "id_registro": df["id_registro"],
        "estado_fips": df["_STATE"].astype("int16"),
        "sexo": categoria(df["SEXVAR"], SEXO),
        "edad": df["_AGE80"].astype("int16"),
        "educacion": categoria(df["EDUCA"], EDUCACION),
        "situacion_laboral": categoria(df["EMPLOY1"], SITUACION_LABORAL, ordenada=False),
        "ingresos": categoria(df["INCOME3"], INGRESOS, ORDEN_INGRESOS).fillna("No informa"),
        "actividad_fisica_30d": categoria(df["EXERANY2"], SI_NO),
        "salud_general": categoria(df["GENHLTH"], SALUD),
        "imc": (df["_BMI5"] / 100).round(2),
        "depresion": categoria(df["ADDEPEV3"], SI_NO),
        "peso_muestral": df["_LLCPWT"].astype("float64"),
    })


def crear_variables(df):
    """Agrega las variables derivadas que usa el análisis."""
    df = df.copy()
    df["inactivo"] = df["actividad_fisica_30d"].eq("No").astype(bool)
    df["grupo_edad"] = pd.cut(df["edad"], bins=[17, 24, 34, 44, 54, 64, 80],
                              labels=["18-24", "25-34", "35-44", "45-54", "55-64", "65+"])
    df["categoria_imc"] = pd.cut(df["imc"], bins=[0, 18.5, 25, 30, np.inf], right=False,
                                 labels=["Bajo peso", "Normal", "Sobrepeso", "Obesidad"])
    return df


def validar_brfss(df):
    """Comprueba las reglas de calidad del BRFSS limpio."""
    assert df["id_registro"].is_unique, "id_registro repetido"
    assert not df.duplicated().any(), "hay filas duplicadas"
    assert df["inactivo"].notna().all() and df["inactivo"].dtype == bool
    assert df["edad"].between(18, 80).all(), "edad fuera de rango"
    assert df["imc"].dropna().between(10, 100).all(), "IMC fuera de rango"
    assert (df["peso_muestral"] > 0).all(), "peso muestral no positivo"
    for col in ["sexo", "educacion", "ingresos", "salud_general", "depresion", "actividad_fisica_30d"]:
        assert isinstance(df[col].dtype, pd.CategoricalDtype), f"{col} no es categórica"
    assert df["ingresos"].notna().all(), "ingresos con faltantes"
    print("Validación BRFSS: OK")
    return df

### Pipeline de limpieza

In [8]:
brfss = (
    brfss_raw
    .pipe(registrar, "BRFSS", "0. Crudo")
    .pipe(crear_identificador)
    .pipe(registrar, "BRFSS", "1. Identificador")
    .pipe(reemplazar_codigos_especiales, CODIGOS_ESPECIALES)
    .pipe(registrar, "BRFSS", "2. Códigos especiales a NaN")
    .pipe(excluir_sin_actividad)
    .pipe(registrar, "BRFSS", "3. Sin respuesta de actividad física")
    .pipe(corregir_tipos_y_etiquetas)
    .pipe(registrar, "BRFSS", "4. Tipos y etiquetas")
    .pipe(crear_variables)
    .pipe(registrar, "BRFSS", "5. Variables derivadas")
    .pipe(validar_brfss)
)
excluidos = len(brfss_raw) - len(brfss)
print(f"Registros excluidos por no tener respuesta válida en EXERANY2: {excluidos:,} ({excluidos / len(brfss_raw):.2%})")
brfss.head()

Validación BRFSS: OK
Registros excluidos por no tener respuesta válida en EXERANY2: 1,251 (0.29%)


,id_registro,estado_fips,sexo,edad,educacion,situacion_laboral,ingresos,actividad_fisica_30d,salud_general,imc,depresion,peso_muestral,inactivo,grupo_edad,categoria_imc
0,1,1,Mujer,80,Universidad incompleta,Jubilado,No informa,No,Muy buena,30.47,No,605.427887,True,65+,Obesidad
1,2,1,Mujer,80,Universidad incompleta,Jubilado,No informa,Sí,Muy buena,28.56,Sí,1121.992705,False,65+,Sobrepeso
2,3,1,Mujer,80,Secundaria completa,Jubilado,Menos de 25 mil USD,Sí,Regular,22.31,No,600.963308,False,65+,Normal
3,4,1,Mujer,78,Universidad incompleta,Jubilado,No informa,Sí,Muy buena,27.44,Sí,605.427887,False,65+,Sobrepeso
4,5,1,Mujer,76,Universidad incompleta,Incapacitado para trabajar,50 a 100 mil USD,Sí,Regular,25.85,Sí,281.711042,False,65+,Sobrepeso


In [9]:
print(brfss.dtypes)
print()
print("Faltantes después de la limpieza:")
print(brfss.isna().sum()[lambda s: s > 0])

id_registro                int32
estado_fips                int16
sexo                    category
edad                       int16
educacion               category
situacion_laboral       category
ingresos                category
actividad_fisica_30d    category
salud_general           category
imc                      float64
depresion               category
peso_muestral            float64
inactivo                    bool
grupo_edad              category
categoria_imc           category
dtype: object

Faltantes después de la limpieza:
educacion             2249
situacion_laboral     7560
salud_general         1200
imc                  40270
depresion             2509
categoria_imc        40270
dtype: int64


**Tratamiento de faltantes:** `imc` conserva sus faltantes (no se imputa, para no inventar datos de salud) y los análisis del IMC
usan solo los registros disponibles. En `ingresos` los faltantes y los códigos 77/99 pasan a la categoría **"No informa"**,
para no perder los 78 548 registros con código 77/99 y los 8 075 faltantes del archivo original. `educacion`, `situacion_laboral`, `salud_general` y `depresion` conservan sus
pocos faltantes como `NaN`.

## Dataset 2: actividad física en Bogotá

In [10]:
FRECUENCIAS = {
    "No practicó deporte ni tuvo actividad física en el mes": "Ninguna en el mes",
    "Menos de 1 vez a la semana": "Menos de 1 vez por semana",
    "1 a 2 veces por semana": "1 a 2 veces por semana",
    "3 o más veces por semana": "3 o más veces por semana",
}


def estandarizar_columnas(df):
    """Nombres sin tildes, en minúsculas y con nombres claros."""
    df = df.rename(columns=normalizar_nombre)
    return df.rename(columns={"ano": "anio", "percepcion": "frecuencia"})


def convertir_porcentaje(df):
    """'36,7' (texto con coma decimal) -> 36.7 (número)."""
    df = df.copy()
    df["porcentaje"] = pd.to_numeric(df["porcentaje"].str.replace(",", ".", regex=False))
    return df


def estandarizar_frecuencia(df):
    """Etiquetas cortas y ordenadas de menor a mayor frecuencia."""
    df = df.copy()
    df["frecuencia"] = pd.Categorical(df["frecuencia"].map(FRECUENCIAS),
                                      categories=list(FRECUENCIAS.values()), ordered=True)
    return df


def separar_total(df):
    """Marca la fila del total de la ciudad para no mezclarla con las localidades."""
    df = df.copy()
    df["es_total_ciudad"] = df["localidad"].str.contains("Bogot", case=False)
    df["anio"] = df["anio"].astype("int16")
    return df


def validar_bogota(df):
    assert df.notna().all().all(), "hay valores faltantes"
    assert df["porcentaje"].between(0, 100).all(), "porcentaje fuera de 0-100"
    assert df["frecuencia"].notna().all(), "frecuencia sin estandarizar"
    assert df["es_total_ciudad"].groupby(df["anio"]).sum().eq(4).all(), "cada año debe tener 4 filas del total"
    assert not df.duplicated(["anio", "localidad", "frecuencia"]).any(), "combinaciones repetidas"
    suma = df.groupby(["anio", "localidad"])["porcentaje"].sum()
    print(f"Suma de las 4 frecuencias por localidad y año: entre {suma.min():.1f} y {suma.max():.1f}")
    # Tolerancia hasta 103: en la fuente, el total "Bogotá D.C." de 2021 suma 102,6 (documentado en data_quality.md)
    assert suma.between(98, 103).all(), "las frecuencias no suman aproximadamente 100"
    print("Validación Bogotá: OK")
    return df

In [11]:
bogota = (
    bogota_raw
    .pipe(registrar, "Bogotá", "0. Crudo")
    .pipe(estandarizar_columnas)
    .pipe(limpiar_texto, ["localidad", "frecuencia", "porcentaje"])
    .pipe(registrar, "Bogotá", "1. Nombres y texto")
    .pipe(convertir_porcentaje)
    .pipe(estandarizar_frecuencia)
    .pipe(separar_total)
    .pipe(registrar, "Bogotá", "2. Tipos y categorías")
    .pipe(validar_bogota)
)
print(bogota.dtypes)
bogota.head()

Suma de las 4 frecuencias por localidad y año: entre 99.9 y 102.6
Validación Bogotá: OK
anio                  int16
localidad            string
frecuencia         category
porcentaje          Float64
es_total_ciudad     boolean
dtype: object


,anio,localidad,frecuencia,porcentaje,es_total_ciudad
0,2017,Usaquén,Ninguna en el mes,36.7,False
1,2017,Chapinero,Ninguna en el mes,30.5,False
2,2017,Santa Fe,Ninguna en el mes,46.1,False
3,2017,San Cristóbal,Ninguna en el mes,56.3,False
4,2017,Usme,Ninguna en el mes,57.5,False


## Dataset 3: Eurobarómetro 97.3

Del archivo de 501 columnas se conservan el identificador, el país, el sexo (`d10`), la edad (`d11`), la frecuencia de práctica
(`qb1`) y las barreras (`qb14_1` a `qb14_13`). Los códigos se traducen con las etiquetas de valor del propio archivo.

In [12]:
BARRERAS_COLS = sorted([c for c in euro_raw.columns if c.startswith("qb14_")], key=lambda c: int(c.split("_")[1]))
COLUMNAS_EURO = ["uniqid", "isocntry", "d10", "d11", "qb1"] + BARRERAS_COLS
print("Columnas no encontradas:", [c for c in COLUMNAS_EURO if c not in euro_raw.columns] or "ninguna")
print("qb1 -", ETIQUETAS_VARIABLES.get("qb1"))
print("   códigos:", ETIQUETAS_VALORES.get("qb1"))
print("d10 códigos:", ETIQUETAS_VALORES.get("d10"))
print("qb14_1 códigos:", ETIQUETAS_VALORES.get("qb14_1"))
pd.Series({c: ETIQUETAS_VARIABLES.get(c) for c in BARRERAS_COLS}, name="etiqueta").to_frame()

Columnas no encontradas: ninguna
qb1 - SPORT ACTIVITY - FREQUENCY
   códigos: {1: '5 times a week or more', 2: '3 to 4 times a week', 3: '1 to 2 times a week', 4: '1 to 3 times a month', 5: 'Less often', 6: 'Never', 7: 'DK (SPONT.)'}
d10 códigos: {1: 'Man', 2: 'Woman', 3: 'None of the above / Non binary / Do not recognize yourself in above categories'}
qb14_1 códigos: {0: 'Not mentioned', 1: 'You do not have the time'}


,etiqueta
qb14_1,NO SPORT ACTIVITY REASON: NO TIME
qb14_2,NO SPORT ACTIVITY REASON: TOO EXPENSIVE
qb14_3,NO SPORT ACTIVITY REASON: COMPETITIVITY
qb14_4,NO SPORT ACTIVITY REASON: NO INFRASTRUCTURE
qb14_5,NO SPORT ACTIVITY REASON: DISABILITY/ILLNESS
qb14_6,NO SPORT ACTIVITY REASON: NO FRIENDS
qb14_7,NO SPORT ACTIVITY REASON: FEEL DISCRIMINATED
qb14_8,NO SPORT ACTIVITY REASON: NO OPPORTUNITY W SAM...
qb14_9,NO SPORT ACTIVITY REASON: LACK MOTIVATION
qb14_10,NO SPORT ACTIVITY REASON: RISK OF INJURIES


In [13]:
# Clasificación de cada barrera según palabras clave de su etiqueta (en inglés en el archivo original)
REGLAS_BARRERAS = [
    (("time",), "Falta de tiempo"),
    (("motivation", "interest"), "Falta de motivación o interés"),
    (("expensive", "cost"), "Costo"),
    (("disab", "illness"), "Discapacidad o enfermedad"),
    (("injur",), "Miedo a lesionarse"),
    (("facilit", "infrastructure", "place"), "Falta de instalaciones cercanas"),
    (("friend", "company", "partner", "someone"), "No tener con quién practicar"),
    (("competit",), "No le gustan las actividades competitivas"),
    (("discrimin",), "Se siente discriminado"),
    (("gender",), "Falta de oportunidades con personas del mismo género"),
    (("other",), "Otra razón"),
    (("none", "nothing", "already"), "Ninguna"),  # "already doing": ya practica deporte, no es una barrera
    (("dk", "don't know", "don’t know"), "No sabe"),
]


def nombre_barrera(etiqueta):
    """Traduce la etiqueta de una barrera usando la parte final (después del último guion)."""
    texto = str(etiqueta).lower().rsplit(" - ", 1)[-1]
    for claves, nombre in REGLAS_BARRERAS:
        if any(re.search(rf"\b{re.escape(k)}", texto) for k in claves):
            return nombre
    return "Por clasificar"


def frecuencia_qb1(etiqueta):
    texto = str(etiqueta).lower()
    # Etiquetas del ZA7888: agrupación estándar del Eurobarómetro (5+ veces/semana = regularmente;
    # 1 a 4 veces/semana = con cierta regularidad; 1 a 3 veces/mes o menos = rara vez)
    for clave, nombre in [("never", "Nunca"), ("seldom", "Rara vez"), ("some regularity", "Con cierta regularidad"),
                          ("regular", "Regularmente"), ("5 times a week", "Regularmente"),
                          ("3 to 4 times a week", "Con cierta regularidad"), ("1 to 2 times a week", "Con cierta regularidad"),
                          ("1 to 3 times a month", "Rara vez"), ("less often", "Rara vez")]:
        if clave in texto:
            return nombre
    return None  # "no sabe" y otras respuestas -> NaN


def sexo_d10(etiqueta):
    texto = str(etiqueta).lower()
    if "woman" in texto or "female" in texto:
        return "Mujer"
    if "man" in texto or "male" in texto:
        return "Hombre"
    return None


NOMBRES_BARRERAS = {c: nombre_barrera(ETIQUETAS_VARIABLES.get(c, "")) for c in BARRERAS_COLS}
pd.Series(NOMBRES_BARRERAS, name="barrera").to_frame()

,barrera
qb14_1,Falta de tiempo
qb14_2,Costo
qb14_3,No le gustan las actividades competitivas
qb14_4,Falta de instalaciones cercanas
qb14_5,Discapacidad o enfermedad
qb14_6,No tener con quién practicar
qb14_7,Se siente discriminado
qb14_8,Falta de oportunidades con personas del mismo ...
qb14_9,Falta de motivación o interés
qb14_10,Miedo a lesionarse


In [14]:
def seleccionar_columnas(df, columnas):
    return df[[c for c in columnas if c in df.columns]].copy()


def recodificar_respuestas(df):
    """Traduce frecuencia y sexo con las etiquetas del archivo y limpia la edad."""
    df = df.copy()
    mapa_qb1 = {codigo: frecuencia_qb1(texto) for codigo, texto in ETIQUETAS_VALORES.get("qb1", {}).items()}
    df["frecuencia_deporte"] = pd.Categorical(
        df["qb1"].map(mapa_qb1), categories=["Nunca", "Rara vez", "Con cierta regularidad", "Regularmente"], ordered=True)
    if "d10" in df.columns:
        mapa_d10 = {codigo: sexo_d10(texto) for codigo, texto in ETIQUETAS_VALORES.get("d10", {}).items()}
        df["sexo"] = pd.Categorical(df["d10"].map(mapa_d10), categories=["Hombre", "Mujer"])
    if "d11" in df.columns:
        df["edad"] = df["d11"].where(df["d11"].between(15, 98)).astype("Int16")
        df["grupo_edad"] = pd.cut(df["edad"], bins=[14, 24, 39, 54, 98], labels=["15-24", "25-39", "40-54", "55+"])
    return df.drop(columns=[c for c in ["qb1", "d10", "d11"] if c in df.columns])


def recodificar_barreras(df):
    """1 = menciona la barrera, 0 = no la menciona; cualquier otro código -> NaN."""
    df = df.copy()
    for col in BARRERAS_COLS:
        df[col] = df[col].map({1: True, 0: False}).astype("boolean")
    df["responde_barreras"] = df[BARRERAS_COLS].notna().any(axis=1)
    return df


def estandarizar_pais(df):
    """'DE-W' y 'DE-E' -> 'DE': un código por país."""
    df = df.copy()
    df["pais"] = df["isocntry"].astype("string").str.split("-").str[0].str.upper()
    return df.drop(columns="isocntry")


def validar_euro(df):
    assert df["uniqid"].is_unique, "uniqid repetido"
    assert len(df) == len(euro_raw), "se perdieron encuestados"
    for col in BARRERAS_COLS:
        assert set(df[col].dropna().unique()) <= {True, False}, f"{col} tiene valores distintos de True/False"
    assert df["pais"].str.fullmatch(r"[A-Z]{2}").all(), "códigos de país no estandarizados"
    print("Países después de estandarizar:", df["pais"].nunique())
    print("Validación Eurobarómetro: OK")
    return df

In [15]:
euro = (
    euro_raw
    .pipe(registrar, "Eurobarómetro", "0. Crudo")
    .pipe(seleccionar_columnas, COLUMNAS_EURO)
    .pipe(registrar, "Eurobarómetro", "1. Módulo de deporte")
    .pipe(recodificar_respuestas)
    .pipe(recodificar_barreras)
    .pipe(estandarizar_pais)
    .pipe(registrar, "Eurobarómetro", "2. Recodificación")
    .pipe(validar_euro)
)
print(euro.dtypes.value_counts())
euro.head()

Países después de estandarizar: 27
Validación Eurobarómetro: OK
boolean     13
int64        1
category     1
category     1
Int16        1
category     1
bool         1
object       1
Name: count, dtype: int64


,uniqid,qb14_1,qb14_2,qb14_3,qb14_4,qb14_5,qb14_6,qb14_7,qb14_8,qb14_9,qb14_10,qb14_11,qb14_12,qb14_13,frecuencia_deporte,sexo,edad,grupo_edad,responde_barreras,pais
0,20000001,False,False,True,True,True,False,False,False,False,False,False,False,False,Con cierta regularidad,Mujer,63,55+,True,AT
1,20000002,False,False,True,False,False,True,True,True,False,False,False,False,False,Con cierta regularidad,Hombre,48,40-54,True,AT
2,20000003,True,True,False,False,False,True,False,True,False,False,False,False,False,Con cierta regularidad,Mujer,28,25-39,True,AT
3,20000004,False,False,True,True,False,False,False,True,False,True,False,False,False,Rara vez,Hombre,69,55+,True,AT
4,20000005,False,True,False,True,False,False,True,True,False,False,False,False,False,Con cierta regularidad,Mujer,54,40-54,True,AT


## Informe de cambios y comparación antes / después

In [16]:
informe = pd.DataFrame(REGISTRO)
informe

,tabla,paso,filas,columnas,nulos,memoria_mb
0,BRFSS,0. Crudo,433323,13,51596,43.0
1,BRFSS,1. Identificador,433323,14,51596,44.6
2,BRFSS,2. Códigos especiales a NaN,433323,14,151294,44.6
3,BRFSS,3. Sin respuesta de actividad física,432072,14,147527,44.5
4,BRFSS,4. Tipos y etiquetas,432072,12,53788,12.8
5,BRFSS,5. Variables derivadas,432072,15,94058,14.0
6,Bogotá,0. Crudo,168,4,0,0.0
7,Bogotá,1. Nombres y texto,168,4,0,0.0
8,Bogotá,2. Tipos y categorías,168,5,0,0.0
9,Eurobarómetro,0. Crudo,26569,501,1462942,204.7


In [17]:
def perfil(df, nombre, momento):
    return {"dataset": nombre, "momento": momento, "filas": len(df), "columnas": df.shape[1],
            "nulos": int(df.isna().sum().sum()), "tipos": ", ".join(sorted(df.dtypes.astype(str).unique())),
            "memoria_mb": round(df.memory_usage(deep=True).sum() / 1_048_576, 1)}

comparacion = pd.DataFrame([
    perfil(brfss_raw, "BRFSS", "Antes"), perfil(brfss, "BRFSS", "Después"),
    perfil(bogota_raw, "Bogotá", "Antes"), perfil(bogota, "Bogotá", "Después"),
    perfil(euro_raw, "Eurobarómetro", "Antes"), perfil(euro, "Eurobarómetro", "Después"),
])
print("Códigos especiales en el BRFSS antes:", int(contar_codigos_especiales(brfss_raw, CODIGOS_ESPECIALES).sum()),
      "| después: 0 (las variables ahora son categorías con etiquetas)")
with pd.option_context("display.max_colwidth", 60):
    display(comparacion)

Códigos especiales en el BRFSS antes: 99698 | después: 0 (las variables ahora son categorías con etiquetas)


,dataset,momento,filas,columnas,nulos,tipos,memoria_mb
0,BRFSS,Antes,433323,13,51596,float64,43.0
1,BRFSS,Después,432072,15,94058,"bool, category, float64, int16, int32",14.0
2,Bogotá,Antes,168,4,0,"int64, str",0.0
3,Bogotá,Después,168,5,0,"Float64, boolean, category, int16, string",0.0
4,Eurobarómetro,Antes,26569,501,1462942,"float64, int64, object, str",204.7
5,Eurobarómetro,Después,26569,20,69,"Int16, bool, boolean, category, int64, object",2.3


**Interpretación:**

- **BRFSS:** pasa de 433 323 a 432 072 filas porque se excluyen los 1 251 registros (0,29 %) sin respuesta válida en `EXERANY2`; ningún otro registro se elimina, ni siquiera los 252 duplicados. Los 99 698 códigos de "no sabe / no responde" dejan de contarse como valores reales. Por eso los nulos suben de 51 596 a 151 294 en el paso 2: es un cambio esperado, que hace visibles datos que antes estaban disfrazados de números. Después bajan a 53 788 al descartar `_TOTINDA` y `_AGEG5YR` y al pasar los faltantes de ingresos a "No informa". Terminan en 94 058 porque la variable derivada `categoria_imc` repite los faltantes de `imc`. Las 13 columnas `float64` se convierten en categorías, enteros y un booleano (`inactivo`), y la memoria baja de 43,0 a 14,0 MB.
- **Bogotá:** conserva sus 168 filas y 0 nulos. La única columna nueva es `es_total_ciudad`, que separa el total de la ciudad de las localidades. `Porcentaje` pasa de texto a `Float64` y la frecuencia queda como categoría ordenada.
- **Eurobarómetro:** conserva los 26 569 encuestados y pasa de 501 a 20 columnas. La memoria baja de 204,7 a 2,3 MB y los nulos, de 1 462 942 a 69. Las 13 barreras quedan como variables booleanas y los 28 códigos de `isocntry` se unifican en 27 países.

En conjunto, la limpieza no pierde información relevante y deja los tres datasets con tipos correctos, etiquetas en español y reglas de calidad verificadas con `assert`.
